<div style="font-size: 24px; line-height: 1.6;">

# Simpson's Paradox: The Group Flips the Story

## Overall metrics can reverse inside groups

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Takeaway

Functions introduced: `groupby`, `.agg`, `pd.crosstab`, `value_counts`.

**Concept learned: important groups can reverse the headline conclusion.**

</div>

<div style="font-size: 24px; line-height: 1.6;">

### Imports

</div>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

<div style="font-size: 24px; line-height: 1.6;">

## The story

Overall, the treatment appears worse. Inside both risk groups, the treatment performs better. **The group variable flips the story.**

</div>

<div style="font-size: 24px; line-height: 1.6;">

## 1. Load the trial data

The file is an Excel workbook. `pd.read_excel()` opens it the same way `pd.read_csv()` opens a CSV — you point it at a path and get back a DataFrame. (Behind the scenes pandas uses the `openpyxl` engine for `.xlsx` files.)

</div>

In [2]:
trial = pd.read_excel("../data/simpsons_paradox_treatment.xlsx")
trial.head()

,patient_id,risk_group,treatment_arm,success
0,1,low_risk,treatment,True
1,2,low_risk,treatment,True
2,3,low_risk,treatment,True
3,4,low_risk,treatment,True
4,5,low_risk,treatment,True


In [3]:
trial.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   patient_id     2000 non-null   int64
 1   risk_group     2000 non-null   str  
 2   treatment_arm  2000 non-null   str  
 3   success        2000 non-null   bool 
dtypes: bool(1), int64(1), str(2)
memory usage: 81.2 KB


<div style="font-size: 24px; line-height: 1.6;">

## 2. Inspect the key columns

Identify the comparison, the outcome, and the possible confounder.

</div>

In [4]:
trial[["risk_group", "treatment_arm", "success"]].head()

,risk_group,treatment_arm,success
0,low_risk,treatment,True
1,low_risk,treatment,True
2,low_risk,treatment,True
3,low_risk,treatment,True
4,low_risk,treatment,True


<div style="font-size: 24px; line-height: 1.6;">

## 3. Start with counts: `value_counts()`

Counts show whether the groups are balanced.

</div>

In [5]:
trial["treatment_arm"].value_counts()

treatment_arm
treatment    1000
control      1000
Name: count, dtype: int64

In [6]:
trial["risk_group"].value_counts()

risk_group
low_risk     1000
high_risk    1000
Name: count, dtype: int64

<div style="font-size: 24px; line-height: 1.6;">

## 4. Two-way counts with `pd.crosstab()`

Composition matters *before* outcome comparison.

</div>

In [7]:
pd.crosstab(trial["risk_group"], trial["treatment_arm"])

treatment_arm,control,treatment
risk_group,,
high_risk,80,920
low_risk,920,80


<div style="font-size: 24px; line-height: 1.6;">

## 5. The rushed analyst's answer: overall success rate

This is where the rushed analyst starts — and stops too early.

</div>

In [8]:
trial.groupby("treatment_arm")["success"].mean()

treatment_arm
control      0.797
treatment    0.334
Name: success, dtype: float64

<div style="font-size: 24px; line-height: 1.6;">

## 6. Now stratify by the hidden variable

Group by risk group **and** treatment arm.

</div>

In [9]:
trial.groupby(["risk_group", "treatment_arm"])["success"].mean()

risk_group  treatment_arm
high_risk   control          0.300000
            treatment        0.288043
low_risk    control          0.840217
            treatment        0.862500
Name: success, dtype: float64

<div style="font-size: 24px; line-height: 1.6;">

## 7. Use `.agg()` for counts and rates together

A rate without a count is fragile. Show both.

</div>

In [10]:
trial.groupby(["risk_group", "treatment_arm"]).agg(
    patients=("patient_id", "count"),
    success_rate=("success", "mean"),
)

patients  success_rate
risk_group treatment_arm                        
high_risk  control              80      0.300000
           treatment           920      0.288043
low_risk   control             920      0.840217
           treatment            80      0.862500

<div style="font-size: 24px; line-height: 1.6;">

## 8. Make the grouped table readable with `unstack()`

</div>

In [11]:
trial.groupby(["risk_group", "treatment_arm"])["success"].mean().unstack()

treatment_arm,control,treatment
risk_group,,
high_risk,0.300000,0.288043
low_risk,0.840217,0.862500


<div style="font-size: 24px; line-height: 1.6;">

## 9. Normalize the crosstab

What fraction of each treatment arm came from each risk group?

</div>

In [12]:
pd.crosstab(trial["risk_group"], trial["treatment_arm"], normalize="columns")

treatment_arm,control,treatment
risk_group,,
high_risk,0.08,0.92
low_risk,0.92,0.08


<div style="font-size: 24px; line-height: 1.6;">

## The four-step Simpson check

1. Overall result
2. Plausible group variable
3. Result inside groups
4. Group composition

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Discussion

- Which number would be easiest to put in a report?
- Which number would be more honest?
- What is driving the reversal — composition or biology?

</div>